# Inference and Ablations for LLaDA-Inspired BERT Diffusion

This notebook is focused on running iterative denoising generation and small ablation sweeps in Google Colab.

It assumes you want to use either the base `bert-base-uncased` model or a checkpoint you previously fine-tuned.

## 1. Install dependencies and clone the repo

In [ ]:
!pip -q install -U pip
!pip -q install -U transformers datasets torch tqdm

from pathlib import Path
import sys

REPO_URL = "https://github.com/lekkalapudiswetha-work/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM.git"
REPO_DIR = Path("/content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM")

if not REPO_DIR.exists():
    !git clone {REPO_URL}

%cd /content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM
!pip -q install -e .

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cpu requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.
torchvision 0.25.0+cpu requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.
Cloning into 'NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 33 (delta 7), reused 27 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 17.69 KiB | 4.42 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  G

## 2. Imports and runtime config

In [ ]:
from dataclasses import dataclass
import json
import random

import numpy as np
import torch

from llada_bert import AblationRunner, BertMaskedLMWrapper, DiffusionNoiseScheduler, ExperimentConfig, IterativeDenoisingSampler


@dataclass
class InferenceConfig:
    model_name: str = "bert-base-uncased"
    checkpoint_path: str | None = "/content/finetuned-model"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42
    prompt: str = "language models can become more useful when"
    sequence_length: int = 24
    steps: int = 12
    threshold: float = 0.85
    temperature: float = 0.9
    top_k: int = 25
    num_samples: int = 3
    remask_strategy: str = "low_confidence"


cfg = InferenceConfig()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

cfg

InferenceConfig(model_name='bert-base-uncased', checkpoint_path='/content/finetuned-model', device='cpu', seed=42, prompt='language models can become more useful when', sequence_length=24, steps=12, threshold=0.85, temperature=0.9, top_k=25, num_samples=3, remask_strategy='low_confidence')

## 3. Choose which model to run

If you previously fine-tuned a model, set `cfg.checkpoint_path` to that saved directory. Otherwise the notebook will use `bert-base-uncased` directly.

In [ ]:
resolved_model_name = cfg.checkpoint_path or cfg.model_name
print("using model:", resolved_model_name)
print("device:", cfg.device)

using model: /content/finetuned-model
device: cpu


## 4. Single inference run

In [ ]:
wrapper = BertMaskedLMWrapper(model_name=resolved_model_name, device=cfg.device)
scheduler = DiffusionNoiseScheduler(total_steps=cfg.steps, base_threshold=cfg.threshold)
sampler = IterativeDenoisingSampler(
    model=wrapper,
    scheduler=scheduler,
    threshold=cfg.threshold,
    top_k=cfg.top_k,
)

result = sampler.sample(
    prompt=cfg.prompt,
    batch_size=cfg.num_samples,
    sequence_length=cfg.sequence_length,
    steps=cfg.steps,
    temperature=cfg.temperature,
    remask_strategy=cfg.remask_strategy,
)

for i, text in enumerate(result.texts, start=1):
    print(f"sample {i}: {text}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

sample 1: language models can become more useful when the state of the language model is not static, and the state of the
sample 2: language models can become more useful when the grammar of the language is more general, and the grammar is less.
sample 3: language models can become more useful when the language of a system is not available, and the system error is that


## 5. Inspect convergence logs

In [ ]:
result.logger.as_rows()

[{'step': 0,
  'masked_tokens': 45,
  'changed_tokens': 0,
  'mean_confidence': 0.3901,
  'accepted_tokens': 0},
 {'step': 1,
  'masked_tokens': 39,
  'changed_tokens': 6,
  'mean_confidence': 0.3835,
  'accepted_tokens': 0},
 {'step': 2,
  'masked_tokens': 36,
  'changed_tokens': 3,
  'mean_confidence': 0.4651,
  'accepted_tokens': 0},
 {'step': 3,
  'masked_tokens': 30,
  'changed_tokens': 6,
  'mean_confidence': 0.5199,
  'accepted_tokens': 0},
 {'step': 4,
  'masked_tokens': 27,
  'changed_tokens': 3,
  'mean_confidence': 0.607,
  'accepted_tokens': 2},
 {'step': 5,
  'masked_tokens': 24,
  'changed_tokens': 3,
  'mean_confidence': 0.6223,
  'accepted_tokens': 0},
 {'step': 6,
  'masked_tokens': 18,
  'changed_tokens': 6,
  'mean_confidence': 0.6745,
  'accepted_tokens': 0},
 {'step': 7,
  'masked_tokens': 15,
  'changed_tokens': 3,
  'mean_confidence': 0.7581,
  'accepted_tokens': 0},
 {'step': 8,
  'masked_tokens': 12,
  'changed_tokens': 3,
  'mean_confidence': 0.7846,
  'accept

## 6. Run a small ablation grid

This uses the repo's `AblationRunner` to compare different step counts and confidence thresholds.

In [ ]:
runner_config = ExperimentConfig(
    model_name=resolved_model_name,
    device=cfg.device,
    steps=cfg.steps,
    sequence_length=cfg.sequence_length,
    temperature=cfg.temperature,
    threshold=cfg.threshold,
    top_k=cfg.top_k,
    seed=cfg.seed,
    num_samples=2,
    remask_strategy=cfg.remask_strategy,
)

runner = AblationRunner(runner_config)
ablation_grid = {
    "threshold": [0.75, 0.85, 0.92],
    "steps": [8, 12, 16],
}

ablation_results = runner.run_ablation_grid(cfg.prompt, ablation_grid)
len(ablation_results)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

9

## 7. View compact ablation summaries

In [ ]:
compact_rows = []
for item in ablation_results:
    compact_rows.append({
        "threshold": item["config"]["threshold"],
        "steps": item["config"]["steps"],
        "final_mean_confidence": item["summary"].get("final_mean_confidence"),
        "final_masked_tokens": item["summary"].get("final_masked_tokens"),
        "sample_preview": item["texts"][0],
    })

compact_rows

[{'threshold': 0.75,
  'steps': 8,
  'final_mean_confidence': 0.8846976161003113,
  'final_masked_tokens': 0.0,
  'sample_preview': 'language models can become more useful when the language is is used, where the language is is used in the language'},
 {'threshold': 0.75,
  'steps': 12,
  'final_mean_confidence': 0.8968794941902161,
  'final_masked_tokens': 0.0,
  'sample_preview': 'language models can become more useful when the language is used in context, and better when the language is used.'},
 {'threshold': 0.75,
  'steps': 16,
  'final_mean_confidence': 0.8887660503387451,
  'final_masked_tokens': 0.0,
  'sample_preview': 'language models can become more useful when the language is not beingable, or if the language is not the same'},
 {'threshold': 0.85,
  'steps': 8,
  'final_mean_confidence': 0.8930955529212952,
  'final_masked_tokens': 0.0,
  'sample_preview': 'language models can become more useful when the language is represented by..,.,.. the model is'},
 {'threshold': 0.85

## 8. Save results as JSON

In [ ]:
output_path = "/content/ablation_results.json"
with open(output_path, "w") as f:
    json.dump(ablation_results, f, indent=2)

print("saved:", output_path)

saved: /content/ablation_results.json


## 9. Suggested experiments

- compare base BERT against your fine-tuned checkpoint
- increase `sequence_length` and `steps`
- try lower temperature for more stable refinement
- compare `top_k` values such as 10, 25, and 50
- log multiple prompts and compare convergence traces